# Monte Carlo Scenario Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
# plotting settings
primary_color = '#00246B'
secondary_color = '#CADCFC'

# global style
sns.set_style("whitegrid")
plt.rcParams['figure.autolayout'] = True
custom_palette = sns.blend_palette([primary_color, secondary_color], n_colors=5)
sns.set_palette(custom_palette)
line_styles = ['-', '--', '-.', ':', '-']

In [ ]:
# load the scenarios generated by main.py
file_path = '../output/scenarios.csv'
df = pd.read_csv(file_path)
print(df.head())
print(np.shape(df))

## Analysis and Visualization of Scenario Data

In [ ]:
# group by scenario and sum Log Returns
assets = [col for col in df.columns if col not in ['Scenario', 'Step']]

# R_cum = exp(sum(r_t)) - 1
scenario_returns = df.groupby('Scenario')[assets].apply(lambda x: np.exp(x.sum()) - 1)
headers = ["HighGrowth", "Bond", "Gold", "1.5xHighGrowth", "LowGrowth"]
scenario_returns.columns = headers
scenario_returns.head()

In [ ]:
summary_stats = scenario_returns.describe().T
summary_stats

In [ ]:
# plot histograms
scenario_returns.hist(bins=100, figsize=(12, 8))
plt.suptitle('Histograms of monthly Returns')
plt.show()

In [ ]:
# visualization of the simulation paths for the price level
assets = [col for col in df.columns if col not in ['Scenario', 'Step']]

# convert returns to price index (Base 1)
fig, axes = plt.subplots(len(assets), 1, figsize=(20,20), sharex=True)
for i, asset in enumerate(assets):
    # get paths for the asset
    asset_returns = df.pivot(index='Step', columns='Scenario', values=asset)
    price_paths = np.exp(asset_returns.cumsum())
    steps = price_paths.index
    
    # stats for visualization
    p10 = price_paths.quantile(0.05, axis=1) 
    p50 = price_paths.quantile(0.50, axis=1)
    p90 = price_paths.quantile(0.95, axis=1) 
    
    ax = axes[i]
    # plot paths
    sample_paths = price_paths.sample(n=1000, axis=1, random_state=42)
    ax.plot(steps, sample_paths, color=secondary_color, alpha=0.4, linewidth=1, zorder=1)
    # 90% confidence interval
    ax.fill_between(steps, p10, p90, color=primary_color, alpha=0.1, label='90% CI')
    # median
    ax.plot(steps, p50, color=primary_color, linewidth=2, label='Median Projection')
    ax.set_title(f'Monte Carlo Simulation for Asset {asset} (90% CI for prices)')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.xlabel('Forecast Month')
plt.tight_layout()
plt.show()

### Tail Risk Measures

In [ ]:
# VaR and CVaRat 95% confidence
def var_and_cvar(x, alpha=0.05):
    # VaR: threshold where 5% of outcomes are worse
    var = x.quantile(alpha)
    # CVaR: average loss of those worst 5% outcomes
    cvar = x[x <= var].mean()
    return pd.Series({'VaR 95%': var, 'CVaR 95%': cvar})

extended_stats = pd.concat([
    scenario_returns.describe(),
    scenario_returns.apply(var_and_cvar)
])

extended_stats_T = extended_stats.T
extended_stats_T

**Observations (Horizon Risk Metrics):**
- **Terminal Risk Interpretation:** Since we simulate a 1-year horizon, these metrics represent the *final* outcome risk. A VaR of -15% means there is a 5% chance the asset value is down >15% at year-end.
- **Value at Risk (VaR 95%):** The "cutoff" for the worst 5% of scenarios. Asset 4 shows significantly deeper VaR, confirming its high leverage risk.
- **Expected Shortfall (CVaR):** This is the *average* loss when the 5% worst-case actually happens. The large gap between VaR and CVaR for the equity-like assets indicates "fat tails"—when things go wrong, they tend to go *very* wrong.

In [ ]:
# Skewness, Kurtosis, and Probability of Crash
def tail_risk_probabilities(x):
    return pd.Series({
        'Skewness': x.skew(),
        'Excess Kurtosis': x.kurtosis(),
        'Prob(Return < -15%)': (x < -0.15).mean(),
        'Prob(Return < -30%)': (x < -0.30).mean(),
        'Prob(Return < -50%)': (x < -0.50).mean()

    })

tail_stats = scenario_returns.apply(tail_risk_probabilities).T
tail_stats

**Observations:**
- **Extreme Fat Tails (Kurtosis):** Asset 4 (Kurtosis > 135) and Asset 1 show extreme fat tails $\to$ massive outlier events (rallies) happen far more frequently than under normal distribution.
- **Positive Skewness (Asymmetry):** All assets have positive skewness $\to$ downside risks exist, but most extreme simulations were gains, not losses
- **Catastrophe Probability:** Asset 2 is effectively "crash-proof" (0.00% chance of losing >30%). Asset 4 is the most fragile, with a ~2% chance of a catastrophic (>30%) collapse and a ~0.5% chance of losing 50% or more.